In [5]:
import os, sys, gc
sys.path.insert(0, os.path.abspath(".."))
import tempfile

# Helps reduce CUDA fragmentation when memory is tight (must be set before heavy allocations).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from pdf2image import convert_from_path
import torch
import transformers
from transformers import AutoModel, AutoTokenizer
from transformers.cache_utils import DynamicCache

print("transformers:", transformers.__version__)

# Compatibility shim for GOT remote code expecting DynamicCache.seen_tokens.
if not hasattr(DynamicCache, "seen_tokens"):
    DynamicCache.seen_tokens = property(
        lambda self: getattr(self, "_seen_tokens", 0),
        lambda self, value: setattr(self, "_seen_tokens", value),
    )

# Clean up previous runs in the same notebook kernel.
if "GOT_MODEL" in globals():
    del GOT_MODEL
if "GOT_TOKENIZER" in globals():
    del GOT_TOKENIZER
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

MODEL_ID = "stepfun-ai/GOT-OCR2_0"
PREFER_CUDA = True

GOT_TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

load_device = "cuda" if (PREFER_CUDA and torch.cuda.is_available()) else "cpu"
load_dtype = torch.float16 if load_device == "cuda" else torch.float32

try:
    GOT_MODEL = AutoModel.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        device_map=load_device,
        use_safetensors=True,
        torch_dtype=load_dtype,
        pad_token_id=GOT_TOKENIZER.eos_token_id,
    ).eval()
    DEVICE = load_device
except torch.cuda.OutOfMemoryError:
    # Fallback to CPU when 6GB VRAM is not enough.
    gc.collect()
    torch.cuda.empty_cache()
    GOT_MODEL = AutoModel.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        device_map="cpu",
        use_safetensors=True,
        torch_dtype=torch.float32,
        pad_token_id=GOT_TOKENIZER.eos_token_id,
    ).eval()
    DEVICE = "cpu"

print("model device:", DEVICE)

transformers: 5.14.1


Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

model device: cuda


In [6]:
import time
import types

pdf_path = r"C:\Users\bella\Desktop\Pessoal\Estudo\korean-study\data\topik_papers\102nd-topik-test-papers\102nd-TOPIK-II-Reading-Test-Paper.pdf"
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF not found: {pdf_path}")

PAGE_NUMBER = 4
DPI = 110
MAX_IMAGE_SIDE = 1280
MAX_NEW_TOKENS = 768
ALLOW_CPU_FALLBACK = False

# Patch generate once: cap decoding length to keep single-page OCR fast.
if not getattr(GOT_MODEL, "_fast_generate_patched", False):
    _orig_generate = GOT_MODEL.generate
    def _fast_generate(self, *args, **kwargs):
        kwargs["max_new_tokens"] = min(int(kwargs.get("max_new_tokens", MAX_NEW_TOKENS)), MAX_NEW_TOKENS)
        kwargs["no_repeat_ngram_size"] = min(int(kwargs.get("no_repeat_ngram_size", 20)), 8)
        return _orig_generate(*args, **kwargs)
    GOT_MODEL.generate = types.MethodType(_fast_generate, GOT_MODEL)
    GOT_MODEL._fast_generate_patched = True

# Read only one page, and keep image smaller to reduce visual token load.
page_image = convert_from_path(pdf_path, first_page=PAGE_NUMBER, last_page=PAGE_NUMBER, dpi=DPI)[0]
page_image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))
print(f"page size: {page_image.size}")

with tempfile.TemporaryDirectory() as tmp_dir:
    tmp_path = os.path.join(tmp_dir, f"page_{PAGE_NUMBER}.png")
    page_image.save(tmp_path, optimize=True)
    print(tmp_path)

    start = time.perf_counter()
    try:
        with torch.inference_mode():
            text = GOT_MODEL.chat(GOT_TOKENIZER, tmp_path, ocr_type="ocr")
    except torch.cuda.OutOfMemoryError:
        print("CUDA OOM during OCR generation.")
        if not ALLOW_CPU_FALLBACK:
            raise RuntimeError(
                "Sem VRAM suficiente para esse page render. Diminua DPI/MAX_IMAGE_SIDE ou ative ALLOW_CPU_FALLBACK=True."
            )
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        GOT_MODEL = GOT_MODEL.to("cpu")
        with torch.inference_mode():
            text = GOT_MODEL.chat(GOT_TOKENIZER, tmp_path, ocr_type="ocr")

elapsed = time.perf_counter() - start
print(f"elapsed: {elapsed:.1f}s")
print(text)

page size: (905, 1280)
C:\Users\bella\AppData\Local\Temp\tmpjzmmibml\page_4.png
CUDA OOM during OCR generation.


RuntimeError: Sem VRAM suficiente para esse page render. Diminua DPI/MAX_IMAGE_SIDE ou ative ALLOW_CPU_FALLBACK=True.